# Presentation figures: fiducial + range, every tracer/set combination

Two companion panels, repeated for every (tracer, simulation-set)
combination that has real cached output on disk:

1. **The kNN-CDF itself** (per k) vs. scale r.
2. **d(CDF)/d(log r)** vs. scale r -- the same log-derivative notebook
   02's "where the CDF has information" section computes, now shown for
   every combination, not just AGN/LH.

**Convention, the same in every panel below**: a **solid line** is the
*fiducial* curve (one line per k), a **shaded band** is the *range*
(literal min-to-max across every realization in that set -- not a
percentile band, not a bootstrap CI; the full spread actually present in
the cached run). What "fiducial" means depends on the set:

- **CV** (fiducial cosmology/astrophysics, seed varies): fiducial = mean
  of the CV realizations; range = min-max across those same realizations
  -- this *is* the pure cosmic-variance noise floor (notebook 05/07),
  drawn directly rather than summarized to one number.
- **LH** (all 6 parameters vary): fiducial = that tracer's own CV mean
  curve, when a CV run for that tracer exists (AGN, galaxies) -- the
  actual fiducial-parameter curve, not an average over the LH prior,
  which needn't sit at the fiducial point at all. Falls back to the LH
  suite's own mean (printed explicitly when this happens -- currently
  only for BH-mass, which has no CV run yet) when no CV run exists for
  that tracer. Range = min-max across the whole retained LH suite --
  this is *not* noise, it's the full footprint of varying every
  parameter at once, marginalized over the other five and over seed.
- **1P** (one parameter varied at a time): fiducial = that tracer's CV
  mean where available (same reasoning as LH -- CV's ~27-30 realizations
  estimate the fiducial curve far more precisely than the 1P set's own
  single fiducial realization, labeled `"0"`), else the 1P set's own
  shared fiducial row. Range = min-max across every step of every
  parameter -- the full footprint of one-at-a-time perturbations.

**Known gaps, stated up front rather than silently skipped**: BH-mass
and galaxy tracers don't have every set generated yet.
- Galaxy 1P: no such run exists (`galaxy_1p_knn_...` would be its
  filename, distinguishable, just not generated) -- skipped below.
- BH-mass 1P/CV: no such run exists *and*, if one were generated with
  `run_suite(tracer="mass", dir_prefix="1P_"/"CV_")` today, it would
  silently collide with AGN's own `1p_knn_...`/`cv_knn_...` files --
  `src/pipeline.py`'s non-LH filename tagging doesn't encode `tracer`,
  only `dir_prefix` (see `src/pipeline.py:206-209`). Not fixed here
  (out of scope for this notebook), but flagged so it isn't a silent
  trap if BH-mass 1P/CV generation is ever attempted.


In [1]:
import sys
sys.path.insert(0, "..")

import glob
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.sensitivity import infer_layout
from src.onep import parse_1p_label

OUTPUT_DIR = config.OUTPUT_DIR
OUT = Path("figs")
OUT.mkdir(exist_ok=True)

# Okabe-Ito colorblind-safe categorical palette -- same as
# presentation/make_presentation_plots.py, so these figures match it.
BLUE, ORANGE, GREEN = "#0072B2", "#E69F00", "#009E73"
GRAY, INK = "#999999", "#333333"
K_COLORS = [BLUE, GREEN, ORANGE]

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 150,
    "font.size": 12,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})


def savefig(fig, name):
    fig.savefig(OUT / name, bbox_inches="tight")
    plt.close(fig)
    print(f"wrote {OUT / name}")


## Helpers

`fiducial_and_range` implements the convention from the intro;
`plot_pair` draws the two companion panels (kNN-CDF, log-derivative) for
one combination, fiducial solid + range shaded, one color per k.


In [2]:
def log_derivative(curve, rgrid):
    """d(curve)/d(log10 r) -- the same computation notebook 02's
    'where the CDF has information' section uses, applied here to every
    curve (fiducial, lo, hi) rather than just the mean."""
    return np.gradient(curve, np.log10(rgrid))


def fiducial_and_range(summaries, n_k, n_r, set_type, sim_ids=None, fiducial_override=None):
    """
    (fiducial, lo, hi), each shaped (n_k, n_r) -- see the intro markdown
    for what 'fiducial' and 'range' mean per set_type ("LH", "CV", "1P").

    fiducial_override: an (n_k, n_r) curve to use as the fiducial line
    instead of computing one from `summaries` (e.g. that tracer's own CV
    mean, when available) -- the range band is always this set's own
    min-max regardless of where the fiducial line comes from.
    """
    lo = summaries.min(axis=0).reshape(n_k, n_r)
    hi = summaries.max(axis=0).reshape(n_k, n_r)

    if fiducial_override is not None:
        return fiducial_override, lo, hi

    if set_type == "1P":
        assert sim_ids is not None, "1P needs sim_ids to find the shared fiducial row"
        is_fiducial = np.array([parse_1p_label(str(s))[0] is None for s in sim_ids])
        if is_fiducial.any():
            fiducial = summaries[is_fiducial].mean(axis=0).reshape(n_k, n_r)
        else:
            print("  (no '0' fiducial row found in this 1P run -- falling back to the set's own mean)")
            fiducial = summaries.mean(axis=0).reshape(n_k, n_r)
    else:
        fiducial = summaries.mean(axis=0).reshape(n_k, n_r)

    return fiducial, lo, hi


def plot_pair(title, filename, rgrid, kvals, fiducial, lo, hi):
    n_k = len(kvals)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

    for ki in range(n_k):
        color = K_COLORS[ki % len(K_COLORS)]
        ax1.fill_between(rgrid, lo[ki], hi[ki], color=color, alpha=0.18, lw=0, zorder=2)
        ax1.plot(rgrid, fiducial[ki], color=color, lw=2.2, label=f"k={kvals[ki]}", zorder=3)

        d_fid = log_derivative(fiducial[ki], rgrid)
        d_lo = log_derivative(lo[ki], rgrid)
        d_hi = log_derivative(hi[ki], rgrid)
        d_lo, d_hi = np.minimum(d_lo, d_hi), np.maximum(d_lo, d_hi)
        ax2.fill_between(rgrid, d_lo, d_hi, color=color, alpha=0.18, lw=0, zorder=2)
        ax2.plot(rgrid, d_fid, color=color, lw=2.2, label=f"k={kvals[ki]}", zorder=3)

    ax1.set_xscale("log")
    ax1.set_xlabel(r"$r\ [\mathrm{Mpc}/h]$")
    ax1.set_ylabel("kNN-CDF")
    ax1.legend(fontsize=9)

    ax2.set_xscale("log")
    ax2.axhline(0, color=INK, lw=0.8, zorder=1)
    ax2.set_xlabel(r"$r\ [\mathrm{Mpc}/h]$")
    ax2.set_ylabel(r"$d\,\mathrm{CDF}/d\log_{10} r$")
    ax2.legend(fontsize=9)

    fig.suptitle(title, y=1.03)
    fig.tight_layout()
    savefig(fig, filename)


## Discover what's on disk

Same discipline as every notebook here: nothing about which combinations
exist is assumed -- this globs for each and reports what it actually
finds before plotting anything.


In [3]:
patterns = {
    "AGN CV":     (f"cv_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz", "CV"),
    "AGN LH":     (f"agn_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz", "LH"),
    "AGN 1P":     (f"1p_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz", "1P"),
    "BH-mass LH": (f"bhmass_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n*.npz", "LH"),
    "Galaxy CV":  (f"galaxy_cv_knn_snap{config.SNAP}_M{config.GALAXY_MASS_CUT:.0e}_n*.npz", "CV"),
    "Galaxy LH":  (f"galaxy_knn_snap{config.SNAP}_M{config.GALAXY_MASS_CUT:.0e}_n*.npz", "LH"),
}

found = {}
for name, (pattern, set_type) in patterns.items():
    matches = sorted(glob.glob(f"{OUTPUT_DIR}/{pattern}"))
    if matches:
        found[name] = (matches[-1], set_type)
        print(f"{name:>10}: found {matches[-1]}")
    else:
        print(f"{name:>10}: NOT FOUND ({pattern}) -- skipping")

print()
print("Galaxy 1P: no run exists yet (run_galaxy_suite was never called with dir_prefix='1P_') -- skipping.")
print("BH-mass 1P/CV: no run exists, and the current filename scheme can't distinguish it from")
print("               AGN's own 1P/CV files even if one were generated -- see intro markdown -- skipping.")


    AGN CV: found ../outputs/cv_knn_snap50_M1e+06_n286.npz
    AGN LH: found ../outputs/agn_knn_snap50_M1e+06_n286.npz
    AGN 1P: found ../outputs/1p_knn_snap50_M1e+06_n286.npz
BH-mass LH: found ../outputs/bhmass_knn_snap50_M1e+06_n286.npz
 Galaxy CV: found ../outputs/galaxy_cv_knn_snap50_M1e+08_n262.npz
 Galaxy LH: found ../outputs/galaxy_knn_snap50_M1e+08_n262.npz

Galaxy 1P: no run exists yet (run_galaxy_suite was never called with dir_prefix='1P_') -- skipping.
BH-mass 1P/CV: no run exists, and the current filename scheme can't distinguish it from
               AGN's own 1P/CV files even if one were generated -- see intro markdown -- skipping.


## AGN's CV and galaxies' CV first

Computed first (not necessarily plotted first) so their fiducial curves
are available to override AGN's and galaxies' LH/1P fiducial lines,
per the convention above.


In [4]:
agn_cv_fiducial = None
gal_cv_fiducial = None
cv_curves = {}  # name -> (rgrid, kvals, fiducial, lo, hi), for plotting below

if "AGN CV" in found:
    path, set_type = found["AGN CV"]
    data = np.load(path, allow_pickle=True)
    n_k, n_r = infer_layout(data["kvals"], data["rgrid"])
    fiducial, lo, hi = fiducial_and_range(data["summaries"], n_k, n_r, set_type)
    agn_cv_fiducial = fiducial
    cv_curves["AGN CV"] = (data["rgrid"], data["kvals"], fiducial, lo, hi)
    print(f"AGN CV fiducial computed from {data['summaries'].shape[0]} realizations")

if "Galaxy CV" in found:
    path, set_type = found["Galaxy CV"]
    data = np.load(path, allow_pickle=True)
    n_k, n_r = infer_layout(data["kvals"], data["rgrid"])
    fiducial, lo, hi = fiducial_and_range(data["summaries"], n_k, n_r, set_type)
    gal_cv_fiducial = fiducial
    cv_curves["Galaxy CV"] = (data["rgrid"], data["kvals"], fiducial, lo, hi)
    print(f"Galaxy CV fiducial computed from {data['summaries'].shape[0]} realizations")


AGN CV fiducial computed from 27 realizations
Galaxy CV fiducial computed from 27 realizations


## The figures

One figure per available combination -- kNN-CDF and its log-derivative,
side by side, fiducial solid + range shaded.


In [5]:
# CV combinations: plot what was already loaded/computed above -- the
# fiducial IS this set's own mean here, nothing to override.
for name, (rgrid, kvals, fiducial, lo, hi) in cv_curves.items():
    tag = name.lower().replace(" ", "_")
    plot_pair(f"{name}: fiducial (solid) + realization range (shaded)",
              f"{tag}.png", rgrid, kvals, fiducial, lo, hi)


wrote figs/agn_cv.png
wrote figs/galaxy_cv.png


In [6]:
def plot_lh_or_1p(name, fiducial_override_for):
    if name not in found:
        return
    path, set_type = found[name]
    data = np.load(path, allow_pickle=True)
    n_k, n_r = infer_layout(data["kvals"], data["rgrid"])

    override = fiducial_override_for.get(name)
    if override is None and set_type in ("LH", "1P"):
        print(f"{name}: no CV run available for this tracer -- fiducial falls back to this set's own mean")

    fiducial, lo, hi = fiducial_and_range(
        data["summaries"], n_k, n_r, set_type,
        sim_ids=data["sim_ids"], fiducial_override=override,
    )
    tag = name.lower().replace(" ", "_").replace("-", "_")
    fid_note = "CV-fiducial" if override is not None else "self-fiducial"
    plot_pair(f"{name}: {fid_note} (solid) + {set_type} range (shaded)",
              f"{tag}.png", data["rgrid"], data["kvals"], fiducial, lo, hi)


fiducial_override_for = {
    "AGN LH": agn_cv_fiducial,
    "AGN 1P": agn_cv_fiducial,
    "BH-mass LH": None,  # no CV run for this tracer -- self-fiducial fallback
    "Galaxy LH": gal_cv_fiducial,
}

for name in ["AGN LH", "AGN 1P", "BH-mass LH", "Galaxy LH"]:
    plot_lh_or_1p(name, fiducial_override_for)


wrote figs/agn_lh.png
wrote figs/agn_1p.png
BH-mass LH: no CV run available for this tracer -- fiducial falls back to this set's own mean
wrote figs/bh_mass_lh.png
wrote figs/galaxy_lh.png


## Reading this

- The **range band is a literal min-max**, not a statistical interval --
  it's deliberately the full, honest spread actually present in the
  cached run, not a percentile band or a bootstrap CI (those already
  exist elsewhere, e.g. `sensitivity_table`'s `ci_lo`/`ci_hi`). For LH
  and 1P that band mixes real parameter-driven variation with
  seed-to-seed noise; it is not a noise floor for those two (the CV
  panels are the noise floor, shown directly).
- **CV's own range band answers "how much does this wobble for free?"**
  (same content as notebook 05/07's scalar noise floor, drawn as a curve
  instead of one number). **LH's and 1P's range bands answer "how far
  does deliberately varying the parameters push the curve, relative to
  that same fiducial line?"** -- putting both on the same fiducial
  reference line is what makes the CV panel a visual noise floor for the
  LH/1P panels sitting next to it.
- **BH-mass's fiducial line is a self-fallback** (this tracer's own LH
  mean, not a true fiducial-parameter curve) since no BH-mass CV run
  exists -- read its panel as internally consistent, not as directly
  comparable in absolute terms to AGN's/galaxies' CV-anchored panels.
- The log-derivative panels' range bands are the derivative of the
  *lo/hi CDF curves* (elementwise min/max after differentiating), not a
  rigorous propagation of the CDF's own uncertainty through the
  derivative -- a standard, understandable shortcut for a presentation
  figure, not a statistical claim.
